# Compare Qwen VLM Predictions with Manual Annotations

Select model and phrase to compare Qwen predictions with manual annotations.

In [1]:
import pandas as pd
from pathlib import Path

# Configure pandas to display full content without truncation
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)

In [2]:
# Paths
BASE_DIR = Path("/mnt/lustre/work/hein/hof217/stable_diffusion")
MANUAL_DIR = BASE_DIR / "z5-manual_annotations"
QWEN_BASE_DIR = BASE_DIR / "x7-z1-qwen_small/new_qwen_outputs"

# Model mapping
MODELS = ['redcaps', 'cogvlm', 'standard']
MODEL_FILES = {
    'redcaps': 'redcaps_clic.csv',
    'cogvlm': 'cogvlm_clic.csv',
    'standard': 'standard.csv'
}
STRATEGIES = ['direct', 'expert', 'inverse']

In [3]:
def parse_indices(s):
    """Parse semicolon-separated indices."""
    if pd.isna(s) or str(s).strip() == '':
        return set()
    return set(int(x.strip()) for x in str(s).replace(';', ',').split(',') if x.strip().isdigit())

def normalize_answer(ans):
    """Normalize Qwen answer to Yes/No/Ambiguous."""
    if pd.isna(ans) or str(ans).strip() == '':
        return 'Ambiguous'
    # Strip whitespace and remove trailing punctuation (., !, ?, etc.)
    ans = str(ans).strip().lower().rstrip('.,!?;:')
    return 'Yes' if ans == 'yes' else 'No' if ans == 'no' else 'Ambiguous'

def get_qwen_answer(row):
    """Extract Qwen answer from row.
    Checks predicted_label first, falls back to full_output if empty.
    This handles 'direct' strategy where answer is in full_output as 'Yes', 'Yes.', 'No', 'No.'.
    """
    # First try predicted_label
    pred_label = row.get('predicted_label', '')
    if pd.notna(pred_label) and str(pred_label).strip() != '':
        return normalize_answer(pred_label)
    # If predicted_label is empty, use full_output
    full_output = row.get('full_output', '')
    return normalize_answer(full_output)

def infer_starts_at_one(df):
    """Infer whether tensor indices in a loaded Qwen file start at 1."""
    if df is None or df.empty or 'tensor_index' not in df.columns:
        return False
    tensor_indices = pd.to_numeric(df['tensor_index'], errors='coerce').dropna()
    if tensor_indices.empty:
        return False
    return int(tensor_indices.min()) >= 1

def index_within_checked(idx, checked, starts_at_one):
    """Check whether idx falls within the manually checked subset."""
    if starts_at_one:
        return 1 <= idx <= checked
    return 0 <= idx < checked

def get_manual_label(idx, data, starts_at_one=False):
    """Get manual label for tensor index."""
    if data is None or not index_within_checked(idx, data['checked'], starts_at_one):
        return None
    if idx in data['incorrect']:
        return 'No'
    if idx in data['ambiguous']:
        return 'Ambiguous'
    return 'Yes'

def load_manual_annotations(model, phrase):
    """Load manual annotations for model and phrase."""
    df = pd.read_csv(MANUAL_DIR / MODEL_FILES[model])
    row = df[df['phrase'] == phrase]
    if row.empty:
        return None
    row = row.iloc[0]
    if pd.isna(row['checked']):
        return None
    return {
        'tensor_file': row['tensor_file'],
        'ambiguous': parse_indices(row['[tensor_index-es for ambigious images]']),
        'incorrect': parse_indices(row['[tensor_index-es for incorrect images]']),
        'checked': int(row['checked'])
    }

def load_qwen_predictions(model, phrase, strategy):
    """Load Qwen predictions for model, phrase, and strategy."""
    qwen_dir = QWEN_BASE_DIR / model / strategy
    phrase_file_map = {
        'close by': 'closeby',
    }
    phrase_file = phrase_file_map.get(phrase, phrase.replace(' ', '_'))
    files = sorted(qwen_dir.glob(f"qwen_{phrase_file}_*.csv"))
    if not files:
        return pd.DataFrame()
    df = pd.read_csv(files[0])
    return df[df['phrase'] == phrase].copy()

def create_confusion_matrix(y_true, y_pred):
    """Create confusion matrix."""
    labels = ['Yes', 'No', 'Ambiguous']
    matrix = pd.DataFrame(0, index=labels, columns=labels)
    for t, p in zip(y_true, y_pred):
        if t in labels and p in labels:
            matrix.loc[t, p] += 1
    return matrix

## Show Available Phrases

Run this cell first to see what phrases are available for each model.

In [5]:
# print("Available phrases per model:\n")
# for model in MODELS:
#     df = pd.read_csv(MANUAL_DIR / MODEL_FILES[model])
#     phrases = df['phrase'].dropna().unique().tolist()
#     print(f"{model.upper()}:")
#     for p in phrases:
#         print(f"  - {p}")
#     print()

## Run Analysis

**Set your model and phrase below:**

In [4]:
# ===== USER INPUT =====
MODEL = 'standard'     # Choose: 'redcaps', 'cogvlm', or 'standard'
PHRASE = 'above'      # Type the phrase (see available phrases above)
# ======================

print(f"Analyzing: {MODEL} / {PHRASE}\n")

# Load manual annotations
manual_data = load_manual_annotations(MODEL, PHRASE)
if manual_data is None:
    print(f"ERROR: No manual annotations found for {MODEL}/{PHRASE}")
else:
    print(f"Manual annotations loaded: {manual_data['checked']} images checked\n")
    
    # Process each strategy
    for strategy in STRATEGIES:
        print(f"{'='*70}")
        print(f"STRATEGY: {strategy.upper()}")
        print(f"{'='*70}\n")
        
        # Load Qwen predictions
        qwen_df = load_qwen_predictions(MODEL, PHRASE, strategy)
        
        if qwen_df.empty:
            print(f"No Qwen predictions found for {strategy}\n")
            continue
        
        starts_at_one = infer_starts_at_one(qwen_df)
        index_mode = '1-based (1..checked)' if starts_at_one else '0-based (0..checked-1)'
        print(f"Index mode inferred: {index_mode}")
        
        # Compare predictions
        results = []
        y_true, y_pred = [], []
        
        for _, row in qwen_df.iterrows():
            idx = int(row['tensor_index'])
            qwen_ans = get_qwen_answer(row)
            manual_ans = get_manual_label(idx, manual_data, starts_at_one=starts_at_one)
            
            if manual_ans:
                y_true.append(manual_ans)
                y_pred.append(qwen_ans)
                
                results.append({
                    'tensor_index': idx,
                    'image': f"{manual_data['tensor_file']}[{idx}]",
                    'image_prompt': f"{row.get('class1', '')} {PHRASE} {row.get('class2', '')}".strip(),
                    'manual_annotation': manual_ans,
                    'qwen_answer': qwen_ans,
                    'qwen_full_answer': row.get('full_output', ''),
                    'qwen_prompt_type': strategy,
                    'match': manual_ans == qwen_ans
                })
        
        results_df = pd.DataFrame(results)
        print(f"Compared against manual labels: {len(y_true)}")
        
        # 1. Confusion Matrix
        if y_true:
            conf_matrix = create_confusion_matrix(y_true, y_pred)
            print("CONFUSION MATRIX (Rows=Manual, Cols=Qwen):")
            print(conf_matrix)
            print()
        
        # 2. Accuracy
        if y_true:
            accuracy = sum([t == p for t, p in zip(y_true, y_pred)]) / len(y_true) * 100
            print(f"ACCURACY: {accuracy:.2f}% ({sum([t == p for t, p in zip(y_true, y_pred)])}/{len(y_true)})\n")
        
        # 3. Mismatch Breakdown
        mismatches = results_df[~results_df['match']]
        if not mismatches.empty:
            print(f"MISMATCHES ({len(mismatches)} total):")
            # display(mismatches[['image', 'image_prompt', 'manual_annotation', 'qwen_answer', 
                            #    'qwen_full_answer', 'qwen_prompt_type']])
        else:
            print("No mismatches!\n")
        
        # 4. All Predictions Summary
        all_qwen = qwen_df.copy()
        all_qwen['qwen_normalized'] = all_qwen.apply(get_qwen_answer, axis=1)
        counts = all_qwen['qwen_normalized'].value_counts()
        total = len(all_qwen)
        print(f"\nALL QWEN PREDICTIONS (Total: {total}):")
        for label in ['Yes', 'No', 'Ambiguous']:
            cnt = counts.get(label, 0)
            print(f"  {label}: {cnt} ({cnt/total*100:.1f}%)")
        print("\n")

Analyzing: standard / above

Manual annotations loaded: 100 images checked

STRATEGY: DIRECT



Index mode inferred: 1-based (1..checked)
Compared against manual labels: 100
CONFUSION MATRIX (Rows=Manual, Cols=Qwen):
           Yes  No  Ambiguous
Yes         24  32          0
No           0  34          0
Ambiguous    0  10          0

ACCURACY: 58.00% (58/100)

MISMATCHES (42 total):

ALL QWEN PREDICTIONS (Total: 100):
  Yes: 24 (24.0%)
  No: 76 (76.0%)
  Ambiguous: 0 (0.0%)


STRATEGY: EXPERT

Index mode inferred: 1-based (1..checked)
Compared against manual labels: 100
CONFUSION MATRIX (Rows=Manual, Cols=Qwen):
           Yes  No  Ambiguous
Yes         48   7          1
No           9  23          2
Ambiguous    1   8          1

ACCURACY: 72.00% (72/100)

MISMATCHES (28 total):

ALL QWEN PREDICTIONS (Total: 100):
  Yes: 58 (58.0%)
  No: 38 (38.0%)
  Ambiguous: 4 (4.0%)


STRATEGY: INVERSE

Index mode inferred: 1-based (1..checked)
Compared against manual labels: 100
CONFUSION MATRIX (Rows=Manual, Cols=Qwen):
           Yes  No  Ambiguous
Yes         46   3          7
No      

In [7]:
OUTPUT_DIR = MANUAL_DIR
LABELS = ['Yes', 'No', 'Ambiguous']
STRATEGY_ORDER = {name: index for index, name in enumerate(STRATEGIES)}
LABEL_ORDER = {name: index for index, name in enumerate(LABELS)}


def export_confusion_matrices_for_model(model):
    """Export one readable CSV per model with all phrase/strategy confusion matrices."""
    manual_df = pd.read_csv(MANUAL_DIR / MODEL_FILES[model])
    export_rows = []
    summary_rows = []

    for _, manual_row in manual_df.iterrows():
        phrase = manual_row['phrase']
        manual_data = load_manual_annotations(model, phrase)
        if manual_data is None:
            continue

        for strategy in STRATEGIES:
            qwen_df = load_qwen_predictions(model, phrase, strategy)
            if qwen_df.empty:
                continue

            starts_at_one = infer_starts_at_one(qwen_df)
            y_true, y_pred = [], []
            for _, row in qwen_df.iterrows():
                idx = int(row['tensor_index'])
                manual_ans = get_manual_label(idx, manual_data, starts_at_one=starts_at_one)
                if manual_ans is None:
                    continue
                y_true.append(manual_ans)
                y_pred.append(get_qwen_answer(row))

            if not y_true:
                continue

            conf_matrix = create_confusion_matrix(y_true, y_pred)
            total_compared = len(y_true)
            correct_predictions = int(sum(t == p for t, p in zip(y_true, y_pred)))
            accuracy_pct = round(correct_predictions / total_compared * 100, 2)

            for manual_label in LABELS:
                export_rows.append({
                    'phrase': phrase,
                    'strategy': strategy,
                    'manual_label': manual_label,
                    'qwen_yes': int(conf_matrix.loc[manual_label, 'Yes']),
                    'qwen_no': int(conf_matrix.loc[manual_label, 'No']),
                    'qwen_ambiguous': int(conf_matrix.loc[manual_label, 'Ambiguous']),
                    'row_total': int(conf_matrix.loc[manual_label].sum()),
                    'total_compared': total_compared,
                    'correct_predictions': correct_predictions,
                    'accuracy_pct': accuracy_pct,
                    'tensor_file': manual_data['tensor_file'],
                })

            summary_rows.append({
                'model': model,
                'phrase': phrase,
                'strategy': strategy,
                'total_compared': total_compared,
                'correct_predictions': correct_predictions,
                'accuracy_pct': accuracy_pct,
            })

    export_df = pd.DataFrame(export_rows)
    if export_df.empty:
        return None, pd.DataFrame()

    export_df['_strategy_order'] = export_df['strategy'].map(STRATEGY_ORDER)
    export_df['_label_order'] = export_df['manual_label'].map(LABEL_ORDER)
    export_df = export_df.sort_values(
        by=['phrase', '_strategy_order', '_label_order'],
        kind='stable'
    ).drop(columns=['_strategy_order', '_label_order'])

    output_path = OUTPUT_DIR / f'confusion_matrices_{model}.csv'
    export_df.to_csv(output_path, index=False)

    return output_path, pd.DataFrame(summary_rows)


created_files = []
summary_frames = []

for model in MODELS:
    output_path, summary_df = export_confusion_matrices_for_model(model)
    if output_path is not None:
        created_files.append(str(output_path))
        summary_frames.append(summary_df)

print('Created files:')
for path in created_files:
    print(f'  - {path}')

if summary_frames:
    summary_df = pd.concat(summary_frames, ignore_index=True)
    summary_df = summary_df.sort_values(by=['model', 'phrase', 'strategy'], kind='stable')
    # print('\nExport summary:')
    # display(summary_df)
else:
    print('No confusion matrices were exported.')

Created files:
  - /mnt/lustre/work/hein/hof217/stable_diffusion/z5-manual_annotations/confusion_matrices_redcaps.csv
  - /mnt/lustre/work/hein/hof217/stable_diffusion/z5-manual_annotations/confusion_matrices_cogvlm.csv
  - /mnt/lustre/work/hein/hof217/stable_diffusion/z5-manual_annotations/confusion_matrices_standard.csv


## Run Analysis for Old Combined CSV Files

Use this when an old CSV already contains all strategies (`direct`, `expert`, `inverse`) in one file with columns like `sd_model_type,tensor_file,tensor_index,phrase,strategy,full_output,predicted_label,class1,class2`.

In [13]:
# # ===== USER INPUT (OLD COMBINED CSV) =====
# MODEL_OLD = 'cogvlm'  # Choose: 'redcaps', 'cogvlm', or 'standard'
# PHRASE_OLD = 'between'  # Type the phrase
# OLD_CSV_PATH = BASE_DIR / 'x7-z1-qwen_small/new_qwen_outputs/qwen_cogvlm_between_inverse_expert_direct_20260303_1919.csv'
# # ==========================================

# print(f"Analyzing old combined CSV: {OLD_CSV_PATH}")
# print(f"Target: {MODEL_OLD} / {PHRASE_OLD}\n")

# manual_data_old = load_manual_annotations(MODEL_OLD, PHRASE_OLD)
# if manual_data_old is None:
#     print(f"ERROR: No manual annotations found for {MODEL_OLD}/{PHRASE_OLD}")
# elif not OLD_CSV_PATH.exists():
#     print(f"ERROR: CSV not found: {OLD_CSV_PATH}")
# else:
#     old_df = pd.read_csv(OLD_CSV_PATH)

#     # Keep only rows for selected model/phrase to match the new analysis behavior.
#     old_df = old_df[
#         (old_df['phrase'] == PHRASE_OLD)
#         & (old_df['sd_model_type'].str.lower() == MODEL_OLD.lower())
#     ].copy()

#     if old_df.empty:
#         print("No rows found in old CSV for the selected model/phrase.")
#     else:
#         available_strategies = [s for s in STRATEGIES if s in set(old_df['strategy'].dropna().str.lower())]
#         print(f"Manual annotations loaded: {manual_data_old['checked']} images checked")
#         print(f"Found strategies in CSV: {available_strategies}\n")

#         for strategy in available_strategies:
#             print(f"{'='*70}")
#             print(f"STRATEGY: {strategy.upper()}")
#             print(f"{'='*70}\n")

#             qwen_df_old = old_df[old_df['strategy'].str.lower() == strategy].copy()

#             if qwen_df_old.empty:
#                 print(f"No predictions found for {strategy}\n")
#                 continue

#             results = []
#             y_true, y_pred = [], []

#             for _, row in qwen_df_old.iterrows():
#                 idx = int(row['tensor_index'])
#                 qwen_ans = get_qwen_answer(row)
#                 manual_ans = get_manual_label(idx, manual_data_old)

#                 if manual_ans:
#                     y_true.append(manual_ans)
#                     y_pred.append(qwen_ans)

#                     results.append({
#                         'tensor_index': idx,
#                         'image': f"{manual_data_old['tensor_file']}[{idx}]",
#                         'image_prompt': f"{row.get('class1', '')} {PHRASE_OLD} {row.get('class2', '')}".strip(),
#                         'manual_annotation': manual_ans,
#                         'qwen_answer': qwen_ans,
#                         'qwen_full_answer': row.get('full_output', ''),
#                         'qwen_prompt_type': strategy,
#                         'match': manual_ans == qwen_ans
#                     })

#             results_df = pd.DataFrame(results)

#             # 1. Confusion Matrix
#             if y_true:
#                 conf_matrix = create_confusion_matrix(y_true, y_pred)
#                 print("CONFUSION MATRIX (Rows=Manual, Cols=Qwen):")
#                 print(conf_matrix)
#                 print()

#             # 2. Accuracy
#             if y_true:
#                 correct = sum([t == p for t, p in zip(y_true, y_pred)])
#                 accuracy = correct / len(y_true) * 100
#                 print(f"ACCURACY: {accuracy:.2f}% ({correct}/{len(y_true)})\n")

#             # 3. Mismatch Breakdown
#             mismatches = results_df[~results_df['match']]
#             if not mismatches.empty:
#                 print(f"MISMATCHES ({len(mismatches)} total):")
#                 display(mismatches[['image', 'image_prompt', 'manual_annotation', 'qwen_answer',
#                                    'qwen_full_answer', 'qwen_prompt_type']])
#             else:
#                 print("No mismatches!\n")

#             # 4. All Predictions Summary
#             all_qwen = qwen_df_old.copy()
#             all_qwen['qwen_normalized'] = all_qwen.apply(get_qwen_answer, axis=1)
#             counts = all_qwen['qwen_normalized'].value_counts()
#             total = len(all_qwen)
#             print(f"\nALL QWEN PREDICTIONS (Total: {total}):")
#             for label in ['Yes', 'No', 'Ambiguous']:
#                 cnt = counts.get(label, 0)
#                 print(f"  {label}: {cnt} ({cnt/total*100:.1f}%)")
#             print("\n")